# Natural Language Processing - Assignment 1
### Cross-Cultural Knowledge Evaluation

**Student Name**: (John) Paul Nagle  
**Student ID**: R00065426  
**Model**: Mistral-7B-Instruct-v0.3 (Or )  
**Locales**: US, UK, Chinese, Ethopian

## Setup and Installation

In [3]:
try:
  import google.colab
  IN_COLAB = True
  print("Running in colab...")
  from google.colab import drive
  drive.mount('/content/drive')
  REQS_PATH="/content/drive/MyDrive/Colab Notebooks/nlp_assignment_1/requirements.txt"
except:
  IN_COLAB = False
  print("Running in local environment...")
  REQS_PATH="requirements.txt"


Running in local environment...


In [5]:
# Install required packages
!pip install -r "{REQS_PATH}" -q


## Imports and Configuration

In [31]:
import warnings
import re
import unicodedata
import json
import random
import numpy as np
from typing import Dict,  Optional, Literal

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0
CUDA available: False


## Locale Configuration

Selected locales:
- **en-US** (English-US): High-resource baseline
- **en-GB** (English-UK): High-resource, European locale
- **zh-CN** (Chinese-China): Non-Latin script, major language
- **am-ET** (Amharic-Ethiopia): Low-resource, under-represented locale

In [32]:
# Define locales
LOCALES = {
    'en-US': {
        'code': 'en-US',
        'name': 'English (United States)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'en-GB': {
        'code': 'en-GB',
        'name': 'English (United Kingdom)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'zh-CN': {
        'code': 'zh-CN',
        'name': 'Chinese (China)',
        'language': 'Simplified Chinese',
        'script': 'Han',
        'resource_level': 'high'
    },
    'am-ET': {
        'code': 'am-ET',
        'name': 'Amharic (Ethiopia)',
        'language': 'Amharic',
        'script': 'Ethiopic',
        'resource_level': 'low'
    }
}

print("Configured Locales:")
for locale_code, config in LOCALES.items():
    print(f"  {locale_code}: {config['name']} ({config['script']} script, {config['resource_level']}-resource)")

Configured Locales:
  en-US: English (United States) (Latin script, high-resource)
  en-GB: English (United Kingdom) (Latin script, high-resource)
  zh-CN: Chinese (China) (Han script, high-resource)
  am-ET: Amharic (Ethiopia) (Ethiopic script, low-resource)


## Load mistralai/Mistral-7B-Instruct-v0.3 Model (or TinyLlama/TinyLlama-1.1B-Chat-v1.0 if on cpu)

**Hardware Detection & Model Loading Strategy:**
- GPU available: Use 4-bit quantization with Mistral-7B
- CPU only: Use smaller model (TinyLlama-1.1B) for faster inference


In [33]:
TEMPERATURE = 0.0  # Required for reproducibility
MAX_NEW_TOKENS = 100 # Maximum number of tokens to generate ????

# Configure model loading based on hardware
if torch.cuda.is_available():
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # 7B params
    # GPU: Use 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    print(f"✓ Model {MODEL_NAME} loaded with 4-bit quantization on GPU")
    
else:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 1.1B params, faster on CPU
    # No quantization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for CPU
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    print("✓ Model loaded on CPU (float32)")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model.eval()
print(f"✓ Model: {MODEL_NAME}")
print(f"✓ Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"✓ Temperature: {TEMPERATURE} (deterministic)")
print(f"✓ Max new tokens: {MAX_NEW_TOKENS}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded on CPU (float32)
✓ Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
✓ Device: CPU
✓ Temperature: 0.0 (deterministic)
✓ Max new tokens: 100


## Text Normalization

Robust normalization strategy for multilingual text matching.

In [34]:
class TextNormalizer:
    """Multi-stage text normalization for answer matching."""

    def __init__(self):
        self.normalization_form  = 'NFC'  # Unicode normalization form

    def normalize(self, text: str, locale: Optional[str] = None) -> str:
        """Normalize text for multilingual text matching with some hard-coded locale-specific handling."""
        if not text:
            return ""

        # 1. Unicode normalization (NFC - Canonical Composition)
        text = unicodedata.normalize(self.normalization_form, text)

        # 2. Remove control characters
        text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]', '', text)

        # 3. Normalize line breaks to \n
        text = text.replace('\r\n', '\n').replace('\r', '\n')

        # 4. Locale-specific whitespace normalization
        if locale == 'zh-CN':
            # Chinese: Remove ALL whitespace (Chinese doesn't use spaces between words)
            text = re.sub(r'\s+', '', text)
        else:
            # Other locales: Standard whitespace normalization
            text = re.sub(r'[ \t]+', ' ', text)
            text = re.sub(r'\s+', ' ', text)
            text = re.sub(r'\n{3,}', '\n\n', text)
            text = text.strip()

        # 5. Convert to lower case (safe for all locales)
        text = text.lower()

        # 6. Locale-specific punctuation handling
        if locale == 'zh-CN':
            # Chinese: Remove both Latin and Chinese punctuation
            # Keep Chinese characters, numbers, and basic Latin letters
            text = re.sub(r'[^\w\u4e00-\u9fff]', '', text, flags=re.UNICODE)
        elif locale == 'am-ET':
            # Amharic: Remove punctuation but preserve Ethiopic script
            # Ethiopic Unicode range: \u1200-\u137F
            text = re.sub(r'[^\w\s\u1200-\u137f\'-]', '', text, flags=re.UNICODE)
            text = re.sub(r'\s+', ' ', text).strip()
        else:
            # English: Remove punctuation but keep apostrophes and hyphens for contractions
            text = re.sub(r'[^\w\s\'\-]', '', text)

        return text

# Initialize normalizer
normalizer = TextNormalizer()


## Base Question System
Base class for all question-answering systems with shared generation logic.

In [35]:
from abc import ABC, abstractmethod

class BaseQuestionSystem(ABC):
    """Base class for question-answering systems with shared generation logic."""
    
    def __init__(self, model, tokenizer, temperature=0.0, max_new_tokens=100):
        """Initialize base system with model and generation parameters."""
        self.model = model
        self.tokenizer = tokenizer
        self.temperature = temperature
        self.max_new_tokens = max_new_tokens
    
    def generate_response(self, prompt: str) -> str:
        """Generate response from model given a prompt.
        
        This method contains the common generation logic shared by all systems.
        """
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.model.device)
        
        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                max_length=None,
                do_sample=False,  # Greedy decoding when temperature=0
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode only the new tokens (not the prompt)
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        response = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        return response
    
    @abstractmethod
    def create_prompt(self, *args, **kwargs) -> str:
        """Create prompt for the model. Must be implemented by subclasses."""
        pass
    
    @abstractmethod
    def generate_answer(self, *args, **kwargs) -> Dict:
        """Generate answer for a question. Must be implemented by subclasses."""
        pass
    
    @abstractmethod
    def evaluate_answer(self, *args, **kwargs) -> bool:
        """Evaluate generated answer. Must be implemented by subclasses."""
        pass

print("✓ Base Question System class defined")

✓ Base Question System class defined


## Baseline SAQ System
### Track A: Short Answer Questions (SAQ)
Direct prompting baseline with locale-aware generation.

In [36]:
class BaselineSAQSystem(BaseQuestionSystem):
    """Baseline Short Answer Question system using direct prompting."""
    def __init__(self, model, tokenizer, normalizer, temperature=0.0):
        super().__init__(model, tokenizer, temperature, max_new_tokens=100)
        self.normalizer = normalizer
    
    def create_prompt(self, question: str, locale: str) -> str:
        """ Create a prompt for the model. """
        locale_config = LOCALES.get(locale)
        if not locale_config:
            raise ValueError(f"Unknown locale: {locale}")
        
        # TinyLlama chat format
        if "TinyLlama" in MODEL_NAME:
            prompt = f"""<|system|>
                        Answer the following question about {locale_config['name']} culture and everyday knowledge.
                        Provide a short, direct answer in {locale_config['language']}.</s>
                        <|user|>
                        {question}</s>
                        <|assistant|>
                        """
        else:
            # Mistral/Llama format
            prompt = f"""[INST] Answer the following question about {locale_config['name']} culture and everyday knowledge.
                        Provide a short, direct answer in {locale_config['language']}.

                        Question: {question}

                        Answer: [/INST]
                      """
    
        return prompt
    
    def generate_answer(self, question: str, locale: str) -> Dict:
        """ Generate answer for a question in the specified locale. """
        # Normalize question
        question = self.normalizer.normalize(question)
        
        # Create prompt
        prompt = self.create_prompt(question, locale)
        
        # Generate response using base class method
        answer = self.generate_response(prompt)
        
        # Normalize answer
        answer = self.normalizer.normalize(answer)
        
        return {
            'question': question,
            'locale': locale,
            'answer': answer,
            'raw_output': answer,
            'prompt': prompt
        }
    
    def evaluate_answer(self, generated_answer: str, question_id: str, reference_data: dict, locale: str = None) -> bool:
        """Evaluate if generated answer matches any reference answer."""
        if question_id not in reference_data:
            return False
        
        # Get all acceptable answers for this question
        annotations = reference_data[question_id]['annotations']
        
        # Normalize generated answer with locale-specific handling
        normalized_generated = self.normalizer.normalize(generated_answer, locale=locale)
        
        # Check against all acceptable answers
        for annotation in annotations:
            for reference_answer in annotation['answers']:
                normalized_reference = self.normalizer.normalize(reference_answer, locale=locale)
                
                # Exact match after normalization
                if normalized_generated == normalized_reference:
                    return True
                
                # Substring match (generated contains reference or vice versa)
                if normalized_reference in normalized_generated or \
                   normalized_generated in normalized_reference:
                    return True
        
        return False

# Initialize baseline system
baseline_system = BaselineSAQSystem(
    model=model,
    tokenizer=tokenizer,
    normalizer=normalizer,
    temperature=TEMPERATURE
)

print("✓ Baseline SAQ System initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Locales: {', '.join(LOCALES.keys())}")

✓ Baseline SAQ System initialized
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Temperature: 0.0
  Locales: en-US, en-GB, zh-CN, am-ET


## Baseline MCQ System
### Track B: Multiple Choice Questions (MCQ)
Direct prompting baseline for multiple choice questions with choice extraction.

In [37]:
import json

class BaselineMCQSystem(BaseQuestionSystem):
    """Baseline Multiple Choice Question system using direct prompting."""
    
    def __init__(self, model, tokenizer, temperature=0.0):
        super().__init__(model, tokenizer, temperature, max_new_tokens=50)
    
    def create_prompt(self, csv_prompt: str) -> str:
        """Use the prompt from CSV file (already formatted with JSON requirement)."""
        # The CSV prompt already contains the question, choices, and JSON format instruction
        # Just wrap it in the appropriate model format
        
        if "TinyLlama" in MODEL_NAME:
            prompt = f"""<|system|>
                You are a helpful assistant that answers questions in JSON format.</s>
                <|user|>
                {csv_prompt}</s>
                <|assistant|>
                """
        else:
            # Mistral format
            prompt = f"""[INST] {csv_prompt} [/INST]"""
        
        return prompt
    
    def extract_choice(self, response: str) -> Optional[str]:
        """Extract choice letter from JSON response format: {\"answer_choice\":\"A\"}."""
        if not response:
            return None
        
        # Try to parse as JSON first
        try:
            # Find JSON object in response
            json_match = re.search(r'\{[^}]*"answer_choice"[^}]*\}', response)
            if json_match:
                json_str = json_match.group(0)
                data = json.loads(json_str)
                choice = data.get('answer_choice', '').strip().upper()
                if choice and choice[0] in 'ABCD':
                    return choice[0]
        except (json.JSONDecodeError, KeyError):
            pass
        
        # Fallback: Try to find first occurrence of A, B, C, or D
        match = re.search(r'\b([A-D])\b', response)
        if match:
            return match.group(1)
        
        # Last resort: check if response starts with a letter
        response = response.strip().upper()
        if response and response[0] in 'ABCD':
            return response[0]
        
        return None
    
    def generate_answer(self, csv_prompt: str, choices: Dict[str, str], 
                       locale: str, country: str) -> Dict:
        """Generate answer for MCQ using CSV prompt."""
        # Create prompt from CSV (which already contains question and format instructions)
        prompt = self.create_prompt(csv_prompt)
        
        # Generate response using base class method
        raw_response = self.generate_response(prompt)
        
        # Extract choice
        predicted_choice = self.extract_choice(raw_response)
        
        return {
            'csv_prompt': csv_prompt,
            'choices': choices,
            'locale': locale,
            'country': country,
            'predicted_choice': predicted_choice,
            'raw_response': raw_response,
            'prompt': prompt
        }
    
    def evaluate_answer(self, predicted_choice: str, correct_answer: str) -> bool:
        """Evaluate if predicted choice matches correct answer."""
        if not predicted_choice:
            return False
        return predicted_choice.upper() == correct_answer.upper()

# Initialize MCQ system
mcq_system = BaselineMCQSystem(
    model=model,
    tokenizer=tokenizer,
    temperature=TEMPERATURE
)

print("✓ Baseline MCQ System initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Max new tokens: {mcq_system.max_new_tokens}")

✓ Baseline MCQ System initialized
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Temperature: 0.0
  Max new tokens: 50


## Load MCQ Data

Load multiple choice questions from CSV file with JSON parsing.

In [38]:
import pandas as pd
import json

def load_mcq_data(csv_path: str, num_questions: Optional[int] = None) -> pd.DataFrame:
    """Load MCQ data from CSV."""
    df = pd.read_csv(csv_path)
    
    # Parse JSON columns
    df['choices'] = df['choices'].apply(json.loads)
    df['choice_countries'] = df['choice_countries'].apply(json.loads)
    
    if num_questions:
        df = df.head(num_questions)
    
    return df

# Load MCQ data
if IN_COLAB:
    mcq_df = load_mcq_data('/content/drive/MyDrive/Colab Notebooks/nlp_assignment_1/data/mcq_questions/mc_questions_file-1.csv', num_questions=2000)
else:
    mcq_df = load_mcq_data('data/mcq_questions/mc_questions_file-1.csv', num_questions=20)

print(f"✓ Loaded {len(mcq_df)} MCQ questions")


✓ Loaded 20 MCQ questions


## Evaluate MCQ System

Run evaluation on MCQ dataset and calculate accuracy.

In [39]:
def evaluate_mcq_system(system, df, locale='en-GB'):
    """Evaluate MCQ system on dataset."""
    results = []
    correct = 0
    total = 0
    
    print(f"Evaluating MCQ System on {len(df)} questions...\n")
    print("="*80)
    
    for idx, row in df.iterrows():
        # Generate answer
        result = system.generate_answer(
            csv_prompt=row['prompt'],
            choices=row['choices'],
            locale=locale,
            country=row['country']
        )
        
        # Evaluate
        is_correct = system.evaluate_answer(
            result['predicted_choice'],
            row['answer_idx']
        )
        
        result['correct_answer'] = row['answer_idx']
        result['is_correct'] = is_correct
        result['mcqid'] = row['MCQID']
        result['id'] = row['ID']
        
        results.append(result)
        
        if is_correct:
            correct += 1
        total += 1
        
        # Print progress
        status = "✓" if is_correct else "✗"
        print(f"{idx+1}. {status} [{row['MCQID']}]")
        print(f"   Answer given: {result['predicted_choice']} | Correct answer: {row['answer_idx']}")
        print(f"   Raw response: [{result['raw_response']}]")
        print("-"*80)
    
    accuracy = correct / total if total > 0 else 0
    
    print("\n" + "="*80)
    print(f"MCQ EVALUATION RESULTS")
    print("="*80)
    print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
    print("="*80)
    
    return results, accuracy

# Run evaluation
mcq_results, mcq_accuracy = evaluate_mcq_system(mcq_system, mcq_df, locale='en-GB')

Evaluating MCQ System on 20 questions...

1. ✗ [Al-en-01_0]
   Answer given: C | Correct answer: D
   Raw response: [   A. candy
                    JSON format: {"answer_choice": "candy"}]
--------------------------------------------------------------------------------
2. ✓ [Al-en-01_1]
   Answer given: C | Correct answer: C
   Raw response: [   A. cookie
                    JSON format: {"answer_choice": "cookie"}]
--------------------------------------------------------------------------------
3. ✗ [Al-en-01_2]
   Answer given: A | Correct answer: C
   Raw response: [   A. cookie]
--------------------------------------------------------------------------------
4. ✗ [Al-en-01_3]
   Answer given: A | Correct answer: D
   Raw response: [   A. chocolate paste
                    JSON format: {"answer_choice": "A"}]
--------------------------------------------------------------------------------
5. ✓ [Al-en-01_4]
   Answer given: C | Correct answer: C
   Raw response: [   Answer choice: 

## MCQ Failure Analysis

Analyze incorrect predictions to understand model behavior.

In [40]:
# Analyze failures
failures = [r for r in mcq_results if not r['is_correct']]

print("\n" + "="*80)
print("MCQ FAILURE ANALYSIS")
print("="*80)
print(f"\nTotal failures: {len(failures)}/{len(mcq_results)}\n")

if failures:
    print("-"*80)
    for idx, fail in enumerate(failures, 1):
        print(f"\n{idx}. ID: {fail['mcqid']}")
        print(f"   Country: {fail['country']}")
        print(f"   Question: {fail['csv_prompt']}...")
        print(f"   Choices: {fail['choices']}")
        print(f"   Predicted: {fail['predicted_choice']}")
        print(f"   Correct: {fail['correct_answer']}")
        print(f"   Raw response: [{fail['raw_response']}]")
        print("-"*80)
else:
    print("No failures - perfect accuracy!")

print("\n" + "="*80)


MCQ FAILURE ANALYSIS

Total failures: 12/20

--------------------------------------------------------------------------------

1. ID: Al-en-01_0
   Country: UK
   Question: What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {"answer_choice":""}

A. candy
B. cookie
C. egg
D. fruit

Answer:...
   Choices: {'A': 'candy', 'B': 'cookie', 'C': 'egg', 'D': 'fruit'}
   Predicted: C
   Correct: D
   Raw response: [   A. candy
                    JSON format: {"answer_choice": "candy"}]
--------------------------------------------------------------------------------

2. ID: Al-en-01_2
   Country: UK
   Question: What is a common snack for preschool kids in the UK? Without any explanation, choose only one from the given alphabet choices(e.g., A, B, C). Provide as JSON format: {"answer_choice":""}

A. cookie
B. egg
C. fruit
D. jam sandwiches

Answer:...
   Choices: {'A': 'cookie', 'B

## Load Reference Answers

Load annotated reference answers from BLEND JSON files for evaluation.

In [41]:
import json

# Load reference answers for each locale
def load_reference_answers(locale_code):
    """Load reference answers from JSON file for a locale"""
    locale_to_file = {
        'en-US': 'US_data.json',
        'en-GB': 'UK_data.json',
        'zh-CN': 'China_data.json',
        'am-ET': 'Ethiopia_data.json'
    }
    
    filename = locale_to_file.get(locale_code)
    if not filename:
        return {}
    
    if IN_COLAB:
        filepath = '/content/drive/MyDrive/Colab Notebooks/nlp_assignment_1/data/saq_answers/'
    else:
        filepath = 'data/saq_answers/'

    try:
        with open(f'{filepath}{filename}', 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return {}

# Load all reference data
REFERENCE_ANSWERS = {
    locale: load_reference_answers(locale) 
    for locale in LOCALES.keys()
}

print("✓ Reference answers loaded")
for locale, data in REFERENCE_ANSWERS.items():
    print(f"  {locale}: {len(data)} questions")

✓ Reference answers loaded
  en-US: 500 questions
  en-GB: 500 questions
  zh-CN: 500 questions
  am-ET: 500 questions


## Load Questions with IDs

Load questions along with their IDs for proper evaluation.

In [ ]:
import pandas as pd

def load_questions_with_ids(locale_code, num_questions=10):
    """Load questions with their IDs from CSV file"""
    locale_to_file = {
        'en-US': 'US_questions.csv',
        'en-GB': 'UK_questions.csv',
        'zh-CN': 'China_questions.csv',
        'am-ET': 'Ethiopia_questions.csv'
    }
    
    filename = locale_to_file.get(locale_code)
    if not filename:
        return []

    if IN_COLAB:
        filepath = '/content/drive/MyDrive/Colab Notebooks/nlp_assignment_1/data/saq_questions/'
    else:
        filepath = 'data/saq_questions/'

    try:
        df = pd.read_csv(f'{filepath}{filename}')
        # Return list of (question_id, question_text) tuples
        questions = [
            (row['ID'], row['Question']) 
            for _, row in df.head(num_questions).iterrows()
        ]
        return questions
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return []

# Test loading
test_questions = load_questions_with_ids('en-GB', 3)


✓ Question loading function ready
  Sample: ('Al-en-01', 'What is a common snack for nursery kids in the UK?')


## Evaluate Baseline System

Run full evaluation across all locales and calculate accuracy metrics.

In [43]:
# Evaluate baseline system
NUM_EVAL_QUESTIONS = 10  # Start with 10 questions per locale

results = {locale: {'correct': 0, 'total': 0, 'details': []} for locale in LOCALES.keys()}

print("Evaluating Baseline System:")
print(f"Testing {NUM_EVAL_QUESTIONS} questions per locale\n")
print("="*80)

for locale in LOCALES.keys():
    print(f"\n{'='*80}")
    print(f"Locale: {locale} ({LOCALES[locale]['name']})")
    print(f"{'='*80}\n")
    
    # Load questions with IDs
    questions_with_ids = load_questions_with_ids(locale, NUM_EVAL_QUESTIONS)
    
    for i, (question_id, question_text) in enumerate(questions_with_ids, 1):
        # Generate answer
        result = baseline_system.generate_answer(question_text, locale)
        generated_answer = result['answer']
        
        # Evaluate answer
        is_correct = baseline_system.evaluate_answer(
            generated_answer, 
            question_id, 
            REFERENCE_ANSWERS[locale],
            locale=locale
        )
        
        # Update results
        results[locale]['total'] += 1
        if is_correct:
            results[locale]['correct'] += 1
        
        # Store details
        results[locale]['details'].append({
            'question_id': question_id,
            'question': question_text,
            'generated': generated_answer,
            'correct': is_correct
        })
        
        # Print result
        status = "✓" if is_correct else "✗"
        print(f"{i}. {status} [{question_id}]")
        print(f"   Q: {question_text}")
        print(f"   A: {generated_answer}")
        print("-"*80)

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}")

Evaluating Baseline System:
Testing 10 questions per locale


Locale: en-US (English (United States))

1. ✓ [Al-en-01]
   Q: What is a common snack for preschool kids in the US?
   A: 1 apple slices with almond butter or peanut butter 2 cheese sticks or crackers with hummus or guacamole 3 carrot sticks with dip or hummus 4 popcorn 5 trail mix with nuts seeds and dried fruit 6 yogurt parfait with granola fruit and a drizzle of
--------------------------------------------------------------------------------
2. ✗ [Al-en-02]
   Q: What is a popular food to go with beer in the US?
   A: 1 pizza pizza is a popular food to go with beer in the united states many bars and restaurants offer pizza and beer combos such as pizza with a beer or a pizza with a beer and a side of fries pizza is a classic american dish that is enjoyed by many people and beer is a popular drink that pairs well with pizza
--------------------------------------------------------------------------------
3. ✓ [Al-en-04]
   

## Display Results Summary

Show accuracy metrics per locale and overall performance.

In [44]:
# Print summary
print(f"\n{'='*80}")
print("EVALUATION SUMMARY")
print(f"{'='*80}\n")

overall_correct = sum(r['correct'] for r in results.values())
overall_total = sum(r['total'] for r in results.values())
overall_accuracy = (overall_correct / overall_total * 100) if overall_total > 0 else 0

print(f"{'Locale':<10} {'Correct':<10} {'Total':<10} {'Accuracy':<10}")
print("-"*80)

for locale in LOCALES.keys():
    correct = results[locale]['correct']
    total = results[locale]['total']
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"{locale:<10} {correct:<10} {total:<10} {accuracy:>6.1f}%")

print("-"*80)
print(f"{'Overall':<10} {overall_correct:<10} {overall_total:<10} {overall_accuracy:>6.1f}%")
print("="*80)


EVALUATION SUMMARY

Locale     Correct    Total      Accuracy  
--------------------------------------------------------------------------------
en-US      9          10           90.0%
en-GB      7          10           70.0%
zh-CN      1          10           10.0%
am-ET      0          10            0.0%
--------------------------------------------------------------------------------
Overall    17         40           42.5%


## Analyze Failures

Examine incorrect answers to understand model weaknesses.

In [45]:
# Show failure examples
print("\n" + "="*80)
print("FAILURE ANALYSIS")
print("="*80 + "\n")

total_failures = 0

for locale in LOCALES.keys():
    failures = [d for d in results[locale]['details'] if not d['correct']]
    total_failures += len(failures)
    
    if failures:
        print(f"\n{locale} - {len(failures)} failures:")
        print("-"*80)
        
        for idx, fail in enumerate(failures, 1):
            print(f"\n{idx}. ID: {fail['question_id']}")
            print(f"   Question: {fail['question']}")
            print(f"   Generated: {fail['generated']}")
            
            # Show expected answers
            if fail['question_id'] in REFERENCE_ANSWERS[locale]:
                expected = REFERENCE_ANSWERS[locale][fail['question_id']]['annotations']
                expected_list = [a['answers'][0] for a in expected[:3]]
                print(f"   Expected: {', '.join(expected_list)}")

print(f"\n{'='*80}")
print(f"Total failures across all locales: {total_failures}/{overall_total}")
print(f"{'='*80}")


FAILURE ANALYSIS


en-US - 1 failures:
--------------------------------------------------------------------------------

1. ID: Al-en-02
   Question: What is a popular food to go with beer in the US?
   Generated: 1 pizza pizza is a popular food to go with beer in the united states many bars and restaurants offer pizza and beer combos such as pizza with a beer or a pizza with a beer and a side of fries pizza is a classic american dish that is enjoyed by many people and beer is a popular drink that pairs well with pizza
   Expected: nuts, pretzels, barbeque

en-GB - 3 failures:
--------------------------------------------------------------------------------

1. ID: Al-en-06
   Question: What is a common school cafeteria food in the UK?
   Generated: 1 a common school cafeteria food in the uk is sandwiches sandwiches are a popular choice among students and many schools offer a variety of options including vegetarian and gluten-free options 2 pasta is also a popular choice in school cafe